## The Future Is Not a Feature: A Look-Ahead Bias-Free Evaluation Framework for Startup Success Prediction through Machine Learning

In [1]:
%load_ext autoreload
%autoreload 2

import os
from dotenv import load_dotenv 
import yaml

import pandas as pd
import polars as pl
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer

from src.preprocessing import preprocessDataset,getCompleteDatasetWithTimeWindow, getCompleteDatasetWithoutTimeWindow
from src.utils import plot_correlation_heatmap, to_tensors, get_probs, plot_shap_comparison, compute_wilcoxon_table, set_seed,compare_metrics, compute_permutation_shap, get_split, clear_split_cache, summarize_metrics


from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import lightgbm as lgb
from src.models.MLP import MLP
from tabpfn import TabPFNClassifier


from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, roc_curve, auc,
                             precision_recall_curve, average_precision_score, roc_auc_score)
import wandb
import shap
import matplotlib.pyplot as plt 


import scipy.integrate
if not hasattr(np, 'trapz'):
    np.trapz = scipy.integrate.trapezoid

load_dotenv()

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)


timeWindow=int(config['time_window'])

lastYear=int(config['last_year'])

# Everything prepare_splits needs, in one place: the split geometry and the
# frequency-encoding settings. The encoding is fitted per seed on the training
# split alone, so the threshold below is the only knob that decides which
# categories survive and which are pooled into "Others".
_freq = config['frequency_encoding']
split_kwargs = dict(
    test_size=config['test_size'],
    cache_dir='tmp/splits',
    categorical_columns=_freq['columns'],
    min_frequency=_freq['min_frequency'],
    other_label=_freq['other_label'],
)

shap_store = {}
metrics_store = {}

/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataset creation

Include all the steps of the feature and target engineering starting from the panel dataset. 

The original initial panel cannot be released. In _data/raw/panel_example.csv_  there is an example with few records.

Loading of the initial panel and university ranking

In [ ]:
# initialPanel = pl.read_csv(config['paths']['raw_dataset'], null_values=["NA"])

# university_ranking_path = config['paths']['raw_university_ranking']

## Dataset with time window (no look-ahead bias)
Already available in _data/processed/dataset_window.csv_

In [ ]:
# datasetWithTimeWindow = getCompleteDatasetWithTimeWindow(initialPanel,timeWindow,lastYear)

# dataset_window = preprocessDataset(datasetWithTimeWindow,university_ranking_path)

# dataset_window.write_csv(config["paths"]["dataset_window"])

# print("Dataset with time window saved to:", config['paths']['dataset_window'])

## Dataset without time window (with look ahead bias)
Already available in _data/processed/dataset_nowindow.csv_

In [ ]:
# # need to re-run the previous cell to get dataset_window before running the following code, to ensure that 
# # the firm in the dataset without time window are the same as those in the dataset with time window

# datasetWithNoTimeWindow = getCompleteDatasetWithoutTimeWindow(initialPanel,dataset_window)

# dataset_nowindow = preprocessDataset(datasetWithNoTimeWindow,university_ranking_path,flag_no_time_window=True)

# # to ensure that the firm in the dataset without time window are the same as those in the dataset with time window, we select only those that are present in the dataset with time window (we must keep the same columns and rows in all the experiments to make them comparable)
# dataset_nowindow = dataset_nowindow.select(dataset_window.columns)

# dataset_nowindow.write_csv(config["paths"]["dataset_nowindow"])

# print("Dataset without time window saved to:", config['paths']['dataset_nowindow'])

## Dataset selection:
(choose one)

Bias controlled experiment

In [2]:
dataset = pd.read_csv(config["paths"]["dataset_window"])
tag="window"

No window experiment (look-ahead bias)

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset_nowindow"])
tag="nowindow"

No Team experiment

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset_window"])
dataset = dataset.drop(columns= [
    "Total_People",
    "WorkExp_Idx_Mean",
    "Highest_Degree_Mean",
    "Total_Founders",
    "Percent_Females",
    "Avg_Earliest_Year",
    "HasTop50Institute",
    "Is_Eco",
    "Is_Eng",
    "Is_NS",
    "Is_Hum",
    "Is_SS",
    "Is_Med",
    "Is_Law",
    "Is_IT",
    "WorkExperienceIndex_CEO",
    "Gender_CEO_Female"
])
tag="noteam"

No Competitors experiment

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset_window"])
dataset = dataset.drop(columns=['SimilarityScoreMean', 'N_Competitors', 'Same_Country'])
tag="nocompetitors"

Label leakage only (control experiment)

The `window` and `nowindow` settings differ in three things at once: the features (41 of the 45 columns), the definition of the target, and the base rate (0.321 against 0.405). The two cells below break that apart, so that a gap in the metrics can be attributed to one leak rather than to their sum.

This one keeps the bias-free features, measured at `StartingAge`, and takes the target of the no-window setting ("does the company ever reach the next stage", instead of "within 7 years of `StartingAge`"). Both processed datasets carry the same 30,300 companies and the same columns, so swapping the target on `CompanyID` is exactly the same dataset with the other label definition.

In [ ]:
# Bias-free features + leaked target: isolates the effect of redefining the label.
_features = pd.read_csv(config["paths"]["dataset_window"]).drop(columns="Target")
_labels = pd.read_csv(config["paths"]["dataset_nowindow"])[["CompanyID", "Target"]]
dataset = _features.merge(_labels, on="CompanyID")
tag = "leaklabel"

# The base rate travels with the label definition, and no model here tunes its
# decision threshold, so it is printed as part of the experiment: AUC is the
# metric to read across the label axis, F1/precision/recall move with it.
print(f"{tag}: {len(dataset)} rows | prevalence {dataset['Target'].mean():.3f}")

Feature leakage only (control experiment)

The mirror image: the features of the no-window setting — taken at the company's last observed age, with the cumulative amounts summed over its whole life — against the bias-free target, "reaches the next stage within 7 years of `StartingAge`".

Compared with `window` it isolates the effect of measuring the features after the fact; compared with `nowindow` it isolates the label, from the other corner of the 2x2.

In [ ]:
# Leaked features + bias-free target: isolates the effect of measuring the
# features at the end of the company's observed life instead of at StartingAge.
_features = pd.read_csv(config["paths"]["dataset_nowindow"]).drop(columns="Target")
_labels = pd.read_csv(config["paths"]["dataset_window"])[["CompanyID", "Target"]]
dataset = _features.merge(_labels, on="CompanyID")
tag = "leakfeat"

print(f"{tag}: {len(dataset)} rows | prevalence {dataset['Target'].mean():.3f}")

Feature correlation

In [ ]:
corr_matrix = plot_correlation_heatmap(dataset)

## Split, Imputation and Scaling

Run for each experiment.

One split per evaluation seed (`seeds` in _config/config.yaml_), each with its own frequency encoding, imputer and scaler fitted on its own training set. Splits are cached under _tmp/splits_, so this cell is slow only the first time per experiment.

The cache file name carries a fingerprint of the dataset columns and of the `frequency_encoding` settings, so editing the threshold in _config/config.yaml_ or regenerating the processed CSVs invalidates it on its own. Set `REBUILD_SPLITS = True` to wipe and rebuild this experiment's splits anyway.

In [3]:
ids = dataset['CompanyID'] 
X = dataset.drop(['CompanyID', 'Target'],axis=1)
y = dataset['Target']

# One independent split per evaluation seed: 60% train, 20% validation
# (threshold tuning), 20% test (final evaluation). Frequency encoding, imputer
# and scaler are all refitted on each seed's training set inside get_split.
# Fitting any of them earlier — across seeds, or before the split, as the
# frequency encoding used to be — leaks held-out information into the features.
#
# The KNN imputation costs minutes per split and the sweep revisits each split
# once per model, so get_split caches to tmp/splits (gitignored). Running this
# cell fills the cache; afterwards each run only loads the split it needs.
#
# The cache key includes a fingerprint of the dataset columns and of the
# encoding settings, so a stale entry is never served after a config or dataset
# change. REBUILD_SPLITS is for the other case: rebuilding on purpose, after
# editing prepare_splits itself. (get_split(..., force=True) does the same for a
# single split, without deleting the old files.)
REBUILD_SPLITS = False

if REBUILD_SPLITS:
    removed = clear_split_cache(cache_dir=split_kwargs['cache_dir'], tag=tag)
    print(f"cache cleared for '{tag}': {removed} file(s) removed")

for seed in config['seeds']:
    get_split(X, y, seed, tag=tag, **split_kwargs)
    print(f"seed {seed}: split ready")

_probe = get_split(X, y, config['seeds'][0], tag=tag, **split_kwargs)
print(f"\nTotal: {len(y)} | Train: {len(_probe['y_train'])} | "
      f"Validation: {len(_probe['y_val'])} | Test: {len(_probe['y_test'])} "
      f"| Seeds: {config['seeds']}")

for column, encoding in _probe['encodings'].items():
    kept = {k: v for k, v in encoding.items() if k != split_kwargs['other_label']}
    print(f"{column}: {len(kept)} categories kept, "
          f"{encoding[split_kwargs['other_label']]:.1%} pooled into "
          f"'{split_kwargs['other_label']}'")

seed 1: split ready
seed 2: split ready
seed 3: split ready
seed 4: split ready
seed 5: split ready
seed 12: split ready

Total: 30300 | Train: 18180 | Validation: 6060 | Test: 6060 | Seeds: [1, 2, 3, 4, 5, 12]
HQCountry: 5 categories kept, 42.1% pooled into 'Others'
PrimaryIndustrySector: 4 categories kept, 5.9% pooled into 'Others'


## Sweep creation

In [4]:

entity=os.getenv("entity")
project=os.getenv("project")

# Initialize a new sweep
sweep_id = wandb.sweep(config["sweep_settings"],entity=entity, project=project)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/giulio-casti/.netrc.


Create sweep with ID: lmshl85d
Sweep URL: https://wandb.ai/giuliocasti-universit-degli-studi-di-cagliari/svm/sweeps/lmshl85d



Definition of the training function to be called by wandb agent.

It will be called multiple times by the wandb agent, each time with a different set of hyperparameters defined in the sweep configuration.

The function trains the specified model, evaluates it on the validation and test sets, logs metrics and plots to Weights & Biases, and computes SHAP values for interpretability.

The _tag_ parameter is used to differentiate between different dataset versions (e.g., with or without time window) when storing SHAP values for later comparison.

In [5]:
def make_train(tag=None):
    def train():
        with wandb.init():
            wandb_config = wandb.config

            # Each run draws its own seed from the sweep. It drives BOTH the
            # split and the model's randomness, so the five runs of a model are
            # five independent replications, not five reruns of one partition —
            # which is what makes the mean +- std in the result tables able to
            # separate a real gap between models from split-to-split noise.
            seed = wandb_config.seed

            split = get_split(X, y, seed, tag=tag, **split_kwargs)
            X_train, X_val, X_test = split['X_train'], split['X_val'], split['X_test']
            y_train, y_val, y_test = split['y_train'], split['y_val'], split['y_test']
            X_train_imp = split['X_train_imp']
            X_val_imp = split['X_val_imp']
            X_test_imp = split['X_test_imp']
            X_train_scaled = split['X_train_scaled']
            X_val_scaled = split['X_val_scaled']
            X_test_scaled = split['X_test_scaled']

            # Reproducibility: seed Python, NumPy, PyTorch (CPU+CUDA), cuDNN.
            # Sklearn / LightGBM models also get a random_state explicitly below.
            set_seed(seed)
            
            # --- RANDOM FOREST ---
            if wandb_config.model_type == "rf":
                model = RandomForestClassifier(
                    n_estimators= wandb_config.rf_n_estimators, 
                    max_depth= wandb_config.rf_max_depth,  
                    min_samples_leaf= wandb_config.rf_min_samples_leaf,
                    max_features='log2',
                    class_weight='balanced',  
                    random_state=seed
                )

                model.fit(X_train_imp, y_train)
                preds_train = model.predict(X_train_imp)
                preds_val = model.predict(X_val_imp)
                preds_test = model.predict(X_test_imp)
                probs_train = model.predict_proba(X_train_imp)[:, 1]
                probs_val = model.predict_proba(X_val_imp)[:, 1]
                probs_test = model.predict_proba(X_test_imp)[:, 1]

                labels_train = y_train
                labels_val = y_val
                labels_test = y_test

            # --- LGBM ---
            elif wandb_config.model_type == "lgb":
                # Ratio for scale_pos_weight (80/20 -> 4) 
                ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1]) 
                
                model = lgb.LGBMClassifier(
                    n_estimators= wandb_config.lgb_n_estimators, 
                    learning_rate= wandb_config.learning_rate, 
                    max_depth= wandb_config.lgb_max_depth, 
                    scale_pos_weight=ratio,  
                    min_child_weight= wandb_config.min_child_weight, 
                    subsample= wandb_config.subsample,                
                    colsample_bytree= wandb_config.colsample_bytree, 
                    reg_alpha= wandb_config.reg_alpha, 
                    reg_lambda= wandb_config.reg_lambda,
                    random_state=seed,
                )

                model.fit(X_train_imp, y_train)
                preds_train = model.predict(X_train_imp)
                preds_val = model.predict(X_val_imp)
                preds_test = model.predict(X_test_imp)
                probs_train = model.predict_proba(X_train_imp)[:, 1]
                probs_val = model.predict_proba(X_val_imp)[:, 1]
                probs_test = model.predict_proba(X_test_imp)[:, 1]

                labels_train = y_train
                labels_val = y_val
                labels_test = y_test

            # --- DECISION TREE ---
            elif wandb_config.model_type == "dt":
                model = DecisionTreeClassifier(
                    max_depth=wandb_config.dt_max_depth,
                    min_samples_leaf=wandb_config.dt_min_samples_leaf,
                    min_samples_split=wandb_config.dt_min_samples_split,
                    criterion=wandb_config.dt_criterion,
                    class_weight='balanced',
                    random_state=seed
                )

                model.fit(X_train_imp, y_train)
                preds_train = model.predict(X_train_imp)
                preds_val = model.predict(X_val_imp)
                preds_test = model.predict(X_test_imp)
                probs_train = model.predict_proba(X_train_imp)[:, 1]
                probs_val = model.predict_proba(X_val_imp)[:, 1]
                probs_test = model.predict_proba(X_test_imp)[:, 1]

                labels_train = y_train
                labels_val = y_val
                labels_test = y_test

            # --- LOGISTIC REGRESSION  ---
            elif wandb_config.model_type == "lr":
                model = LogisticRegression(
                    C=wandb_config.lr_C,  
                    penalty=wandb_config.lr_penalty,
                    solver='saga',
                    class_weight='balanced',
                    max_iter=1000,
                    random_state=seed
                )

                model.fit(X_train_scaled, y_train)
                preds_train = model.predict(X_train_scaled)
                preds_val = model.predict(X_val_scaled)
                preds_test = model.predict(X_test_scaled)
                probs_train = model.predict_proba(X_train_scaled)[:, 1]
                probs_val = model.predict_proba(X_val_scaled)[:, 1]
                probs_test = model.predict_proba(X_test_scaled)[:, 1]

                labels_train = y_train
                labels_val = y_val
                labels_test = y_test

            # --- SVM (RBF KERNEL) ---
            elif wandb_config.model_type == "svm":
                model = SVC(
                    C=wandb_config.svm_C,
                    gamma=wandb_config.svm_gamma,
                    kernel='rbf',
                    probability=True,
                    class_weight='balanced',
                    cache_size=1000,
                    random_state=seed
                )

                # Scaled, like LR and the MLP: an RBF kernel compares distances,
                # so unscaled features would let the widest-range column dominate.
                model.fit(X_train_scaled, y_train)
                probs_train = model.predict_proba(X_train_scaled)[:, 1]
                probs_val = model.predict_proba(X_val_scaled)[:, 1]
                probs_test = model.predict_proba(X_test_scaled)[:, 1]

                # Thresholded on the probability rather than taken from
                # model.predict(): for SVC the two disagree, because predict()
                # uses the sign of the decision function while predict_proba()
                # goes through Platt scaling. Every other model here decides at
                # p >= 0.5, and the comparison table is only meaningful if the
                # SVM decides the same way.
                preds_train = (probs_train >= 0.5).astype(int)
                preds_val = (probs_val >= 0.5).astype(int)
                preds_test = (probs_test >= 0.5).astype(int)

                labels_train = y_train
                labels_val = y_val
                labels_test = y_test

            elif wandb_config.model_type == "mlp":
                # ── SWEEP HYPERPARAMETERS ──
                hidden_sizes = [int(x) for x in wandb_config.hidden_sizes.split(",")] 
                mlp_lr = wandb_config.mlp_learning_rate 
                mlp_dropout = wandb_config.dropout_rate 
                mlp_wd = wandb_config.weight_decay
                BATCH_SIZE = wandb_config.batch_size 

                # ── TENSOR CONVERSION ──
                X_train_t, y_train_t = to_tensors(X_train_scaled, y_train)
                X_val_t,   y_val_t   = to_tensors(X_val_scaled,   y_val)
                X_test_t,  y_test_t  = to_tensors(X_test_scaled,  y_test)

                # ── DATALOADER ──
                # Seeded generator so that shuffle order is reproducible across runs.
                train_generator = torch.Generator()
                train_generator.manual_seed(seed)
                train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                                        batch_size=BATCH_SIZE, shuffle=True,
                                        generator=train_generator)
                val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t),
                                        batch_size=BATCH_SIZE)
                test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t),
                                        batch_size=BATCH_SIZE)

                # ── MODEL, LOSS, OPTIMIZER ──
                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

                model = MLP(
                    input_size=X_train_scaled.shape[1],
                    hidden_sizes=hidden_sizes,
                    dropout_rate=mlp_dropout,
                    batch_norm=True
                ).to(device)

                # Class weight: pos_weight = n_neg / n_pos (loss balancing)
                n_pos = y_train.sum()
                n_neg = len(y_train) - n_pos
                pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
                criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

                optimizer = torch.optim.AdamW(model.parameters(), lr=mlp_lr, weight_decay=mlp_wd)
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='max', factor=0.5, patience=5
                )
                # ── TRAINING LOOP ──
                EPOCHS = 100
                best_auc = 0.0
                patience_counter = 0
                EARLY_STOPPING_PATIENCE = 10

                # Temporary path for best model checkpoint
                _tmp_dir = os.path.join(os.getcwd(), "tmp")
                os.makedirs(_tmp_dir, exist_ok=True)
                best_model_path = os.path.join(_tmp_dir, "best_model.pt")

                for epoch in range(EPOCHS):

                    # ── Train ──
                    model.train()
                    train_loss = 0.0

                    for X_batch, y_batch in train_loader:
                        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                        optimizer.zero_grad()
                        preds = model(X_batch)
                        loss  = criterion(preds, y_batch)
                        loss.backward()
                        optimizer.step()

                        train_loss += loss.item() * len(X_batch)

                    train_loss /= len(train_loader.dataset)

                    # ── Validation ──
                    model.eval()
                    val_loss = 0.0
                    all_preds, all_labels = [], []

                    with torch.no_grad():
                        for X_batch, y_batch in val_loader:
                            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                            logits = model(X_batch)
                            loss   = criterion(logits, y_batch)
                            val_loss += loss.item() * len(X_batch)

                            probs = torch.sigmoid(logits).cpu().numpy()
                            all_preds.extend(probs)
                            all_labels.extend(y_batch.cpu().numpy())

                    val_loss /= len(val_loader.dataset)
                    val_auc   = roc_auc_score(all_labels, all_preds)
                    val_f1    = f1_score(all_labels, (np.array(all_preds) > 0.5).astype(int))

                    scheduler.step(val_auc)

                    print(f"Epoch {epoch+1:3d} | "
                        f"Train Loss: {train_loss:.4f} | "
                        f"Val Loss: {val_loss:.4f} | "
                        f"Val AUC: {val_auc:.4f} | "
                        f"Val F1: {val_f1:.4f}")

                    # ── Early Stopping + Best Model ──
                    if val_auc > best_auc:
                        best_auc = val_auc
                        patience_counter = 0
                        torch.save(model.state_dict(), best_model_path)
                    else:
                        patience_counter += 1
                        if patience_counter >= EARLY_STOPPING_PATIENCE:
                            print(f"\nEarly stopping at epoch {epoch+1}. Best AUC: {best_auc:.4f}")
                            break
                # ── EVALUATION ──
                model.load_state_dict(torch.load(best_model_path, weights_only=True))
                if os.path.exists(best_model_path):
                    os.remove(best_model_path)
                model.eval()

                probs_train, labels_train = get_probs(train_loader, model, device)
                probs_val,   labels_val   = get_probs(val_loader,   model, device)
                probs_test,  labels_test  = get_probs(test_loader,  model, device)
                preds_train = (probs_train >= 0.5).astype(int)
                preds_val   = (probs_val   >= 0.5).astype(int)
                preds_test  = (probs_test  >= 0.5).astype(int)

            # --- TABPFN (v2, run locally) ---
            elif wandb_config.model_type == "tabpfn":
                # Same checkpoint the Prior Labs API served as "v2_default";
                # locally model_path is a file name, downloaded to the TabPFN
                # cache on first use.
                #
                # ignore_pretraining_limits: the v2 checkpoint declares a 10k-row
                # pretraining limit and the training split holds ~18k, so without
                # this fit() refuses. The API applied a 50k limit to the same
                # model, so lifting it is what keeps the local run equivalent —
                # and it keeps the train set identical to the one the other
                # models see, instead of subsampling it.
                #
                # device="auto": CUDA when the machine has it, CPU otherwise.
                # TabPFN on CPU past 1000 rows is slow enough to be impractical
                # here, so this model is meant for the GPU server.
                model = TabPFNClassifier(
                    model_path="tabpfn-v2-classifier-v2_default.ckpt",
                    n_estimators=wandb_config.tabpfn_n_estimators,
                    balance_probabilities=True,
                    ignore_pretraining_limits=True,
                    device="auto",
                    random_state=seed
                )

                # Imputed but unscaled, like RF/LGBM/DT: TabPFN normalises its
                # inputs internally and does not want RobustScaler output.
                model.fit(X_train_imp, y_train)
                probs_train = model.predict_proba(X_train_imp)[:, 1]
                probs_val = model.predict_proba(X_val_imp)[:, 1]
                probs_test = model.predict_proba(X_test_imp)[:, 1]

                # Thresholded rather than obtained from model.predict(): for a
                # binary task predict() is argmax(predict_proba), i.e. the same
                # p >= 0.5 rule, so calling it would pay for a second forward
                # pass over the same rows.
                preds_train = (probs_train >= 0.5).astype(int)
                preds_val = (probs_val >= 0.5).astype(int)
                preds_test = (probs_test >= 0.5).astype(int)

                labels_train = y_train
                labels_val = y_val
                labels_test = y_test

            # Test set metrics
            acc       = accuracy_score(labels_test, preds_test)
            acc_train = accuracy_score(labels_train, preds_train)
            precision = precision_score(labels_test, preds_test, zero_division=0)
            recall    = recall_score(labels_test, preds_test, zero_division=0)
            f1        = f1_score(labels_test, preds_test, zero_division=0)
            f1_train  = f1_score(labels_train, preds_train, zero_division=0)
            f1_val    = f1_score(labels_val, preds_val, zero_division=0)

            # ROC Curve (test set)
            fpr, tpr, _ = roc_curve(labels_test, probs_test)
            roc_auc = auc(fpr, tpr)

            fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
            ax_roc.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
            ax_roc.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
            ax_roc.set_xlim([0.0, 1.0])
            ax_roc.set_ylim([0.0, 1.05])
            ax_roc.set_xlabel('False Positive Rate')
            ax_roc.set_ylabel('True Positive Rate')
            ax_roc.set_title(f'ROC - {wandb_config.model_type.upper()}')
            ax_roc.legend(loc='lower right')
            ax_roc.grid(alpha=0.3)

            # Precision-Recall Curve (test set)
            prec_curve, rec_curve, _ = precision_recall_curve(labels_test, probs_test)
            ap = average_precision_score(labels_test, probs_test)

            fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
            ax_pr.plot(rec_curve, prec_curve, color='green', lw=2, label=f'PR curve (AP = {ap:.3f})')
            ax_pr.set_xlim([0.0, 1.0])
            ax_pr.set_ylim([0.0, 1.05])
            ax_pr.set_xlabel('Recall')
            ax_pr.set_ylabel('Precision')
            ax_pr.set_title(f'Precision-Recall - {wandb_config.model_type.upper()}')
            ax_pr.legend(loc='upper right')
            ax_pr.grid(alpha=0.3)

            wandb.log({
                "accuracy_train": acc_train,
                "F1_train": f1_train,
                "F1_val": f1_val,
                "accuracy": acc,
                "F1": f1,
                "precision": precision,
                "recall": recall,
                "AUC": roc_auc,
                "average_precision": ap,
                "roc_curve": wandb.Image(fig_roc),
                "pr_curve": wandb.Image(fig_pr),
            })
            plt.close(fig_roc)
            plt.close(fig_pr)

            print(f"AUC: {roc_auc:.3f} | AP: {ap:.3f}")


            print(classification_report(labels_test, preds_test))


            wandb.sklearn.plot_confusion_matrix(labels_test, preds_test, ["Neg", "Pos"])

            # Persist metrics for later cross-experiment comparison (parallel to shap_store).
            if tag is not None:
                metrics_store.setdefault((wandb_config.model_type, tag), []).append({
                    "seed":              seed,
                    "accuracy_train":    acc_train,
                    "F1_train":          f1_train,
                    "F1_val":            f1_val,
                    "accuracy":          acc,
                    "F1":                f1,
                    "precision":         precision,
                    "recall":            recall,
                    "AUC":               roc_auc,
                    "average_precision": ap,
                })

            # SHAP is computed on the first evaluation seed only. These tables
            # answer a different question from the mean +- std ones: which
            # features move when the window is removed, not how much the metrics
            # vary across splits. compute_wilcoxon_table also pairs rows within
            # one explained sample, so pooling five seeds would change what the
            # test measures. It additionally keeps TabPFN to one SHAP pass per
            # experiment instead of five, which its inference cost notices.
            if seed == config['shap_seed']:
                # Re-seed before SHAP so that the stochastic SHAP estimators
                # (shap.sample, KernelExplainer) give the same values across runs.
                set_seed(seed)

                # SVM and TabPFN are explained with PermutationExplainer, whose cost
                # is n_explain * (2*n_features+1) * n_background rows of prediction.
                # They get one budget each: the SVM pays CPU time, TabPFN pays a
                # forward pass per row on the GPU. See config.yaml.
                shap_cfg = config['shap_permutation'].get(wandb_config.model_type, {})

                # Data consistent with the model used
                if wandb_config.model_type in ["rf", "lgb", "dt"]:
                    explainer_sample = X_test.iloc[:1000]          # non scaled — RF, LGBM, DT fitted on X_train_imp
                elif wandb_config.model_type == "lr":
                    explainer_sample = X_test_scaled[:1000]         # scaled — LR fitted on X_train_scaled
                elif wandb_config.model_type == "mlp":
                    explainer_sample = X_test_scaled[:1000]         # scaled — MLP fitted on X_train_scaled
                elif wandb_config.model_type == "svm":
                    explainer_sample = X_test_scaled[:shap_cfg['n_explain']]   # scaled — SVM fitted on X_train_scaled
                else:
                    explainer_sample = X_test_imp[:shap_cfg['n_explain']]      # imputed, unscaled — TabPFN fitted on X_train_imp

                if wandb_config.model_type in ["rf", "lgb", "dt"]:
                    explainer = shap.TreeExplainer(model)
                    shap_result = explainer.shap_values(explainer_sample)

                    if isinstance(shap_result, list):
                        shap_values = shap_result[1]
                    elif hasattr(shap_result, 'shape') and len(shap_result.shape) == 3:
                        shap_values = shap_result[:, :, 1]
                    else:
                        shap_values = shap_result

                elif wandb_config.model_type == "lr":
                    explainer = shap.LinearExplainer(model, X_train_scaled)
                    shap_values = explainer.shap_values(explainer_sample)

                elif wandb_config.model_type == "mlp":
                    def mlp_predict(x):
                        model.eval()
                        with torch.no_grad():
                            t = torch.tensor(x, dtype=torch.float32).to(device)
                            logits = model(t)
                            probs = torch.sigmoid(logits).cpu().numpy().flatten()
                        return probs

                    background = shap.sample(X_train_scaled, 30, random_state=seed)
                    explainer = shap.KernelExplainer(mlp_predict, background)
                    shap_values = explainer.shap_values(explainer_sample, nsamples=1000)

                else:
                    # SVM and TabPFN: no TreeExplainer, no LinearExplainer (the RBF
                    # kernel is not linear), and KernelExplainer is intractable on
                    # both. PermutationExplainer needs only 2*n_features+1
                    # evaluations per explained row.
                    background = (X_train_scaled if wandb_config.model_type == "svm"
                                  else X_train_imp)
                    shap_values = compute_permutation_shap(
                        model,
                        background,
                        explainer_sample,
                        n_explain=shap_cfg['n_explain'],
                        n_background=shap_cfg['n_background'],
                        random_state=seed,
                    )

                # Summary plot 
                plt.figure(figsize=(10, 6))
                shap.summary_plot(
                    shap_values,
                    explainer_sample,
                    feature_names=X.columns.tolist(),
                    show=False
                )
                wandb.log({"shap_summary_plot": wandb.Image(plt)})
                plt.close()
            
                if(tag is not None):
                    shap_store[(wandb_config.model_type, tag)] = {
                        "shap_values": np.asarray(shap_values),
                        "explainer_sample": explainer_sample,
                    }
    return train
                

# Experiment

## Sweep start

_tag_ is usefull for shap comparison between models, change with "nowindow" if you are running the no window experiment. 

_number_of_runs_ indicate the number of runs for the sweep. 

 - Set 70 and set a fixed model in _config/config.yaml_ (model_type) if you want to search the best hyperparams configuration. Set `seeds: &seeds [12]` in _config/config.yaml_: a single seed, and deliberately not one of the evaluation five, because several seeds would let bayes optimise the seed and report the luckiest split, and tuning on a split you later report on inflates it.

 - Set 35 if you want to test all the models with the best found configuration: set `seeds: &seeds [1, 2, 3, 4, 5]` and the grid crosses the 7 models with those 5 seeds, so each model is replicated five times. Make sure all the models are enabled in _config/config.yaml_ model_type

`seeds` is the only place a seed is written: the sweep block references it (`values: *seeds`) and the split cell prepares exactly those splits, so run the split cell again after changing it.

`svm` and `tabpfn` are part of the sweep. TabPFN (v2) runs locally: the checkpoint is downloaded to the TabPFN cache on first use, and inference needs a CUDA GPU — on CPU it is impractical at this dataset size.

The best hyperparameter configurations for each model are avaible in _config/config.yaml_ 

In [6]:
number_of_runs = 35

wandb.agent(sweep_id, function=make_train(tag), count=number_of_runs)

wandb: Agent Starting Run: ijwb37uo with config:
wandb: 	batch_size: 256
wandb: 	colsample_bytree: 1
wandb: 	dropout_rate: 0.3
wandb: 	dt_criterion: entropy
wandb: 	dt_max_depth: 5
wandb: 	dt_min_samples_leaf: 2
wandb: 	dt_min_samples_split: 9
wandb: 	hidden_sizes: 512,256,128
wandb: 	learning_rate: 0.03092792034710772
wandb: 	lgb_max_depth: 5
wandb: 	lgb_n_estimators: 300
wandb: 	lr_C: 0.29719610165875604
wandb: 	lr_penalty: l2
wandb: 	min_child_weight: 20
wandb: 	mlp_learning_rate: 0.006481694307849973
wandb: 	model_type: rf
wandb: 	reg_alpha: 7.471515240794344
wandb: 	reg_lambda: 3.0246578811237925
wandb: 	rf_max_depth: 15
wandb: 	rf_min_samples_leaf: 9
wandb: 	rf_n_estimators: 200
wandb: 	seed: 1
wandb: 	subsample: 0.4
wandb: 	svm_C: 0.7
wandb: 	svm_gamma: scale
wandb: 	tabpfn_n_estimators: 8
wandb: 	weight_decay: 0.00012295217130781902
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/giulio-casti/.netrc.
wandb: Currently logged in as: giuliocasti (giul

AUC: 0.806 | AP: 0.662
              precision    recall  f1-score   support

           0       0.83      0.79      0.81      4116
           1       0.60      0.67      0.63      1944

    accuracy                           0.75      6060
   macro avg       0.72      0.73      0.72      6060
weighted avg       0.76      0.75      0.75      6060



AUC,▁
F1,▁
F1_train,▁
F1_val,▁
accuracy,▁
accuracy_train,▁
average_precision,▁
precision,▁
recall,▁
AUC,0.80615
F1,0.63178


wandb: Agent Starting Run: 4ficv47d with config:
wandb: 	batch_size: 256
wandb: 	colsample_bytree: 1
wandb: 	dropout_rate: 0.3
wandb: 	dt_criterion: entropy
wandb: 	dt_max_depth: 5
wandb: 	dt_min_samples_leaf: 2
wandb: 	dt_min_samples_split: 9
wandb: 	hidden_sizes: 512,256,128
wandb: 	learning_rate: 0.03092792034710772
wandb: 	lgb_max_depth: 5
wandb: 	lgb_n_estimators: 300
wandb: 	lr_C: 0.29719610165875604
wandb: 	lr_penalty: l2
wandb: 	min_child_weight: 20
wandb: 	mlp_learning_rate: 0.006481694307849973
wandb: 	model_type: rf
wandb: 	reg_alpha: 7.471515240794344
wandb: 	reg_lambda: 3.0246578811237925
wandb: 	rf_max_depth: 15
wandb: 	rf_min_samples_leaf: 9
wandb: 	rf_n_estimators: 200
wandb: 	seed: 2
wandb: 	subsample: 0.4
wandb: 	svm_C: 0.7
wandb: 	svm_gamma: scale
wandb: 	tabpfn_n_estimators: 8
wandb: 	weight_decay: 0.00012295217130781902
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/giulio-casti/.netrc.


AUC: 0.798 | AP: 0.647
              precision    recall  f1-score   support

           0       0.83      0.77      0.80      4116
           1       0.58      0.67      0.62      1944

    accuracy                           0.74      6060
   macro avg       0.71      0.72      0.71      6060
weighted avg       0.75      0.74      0.74      6060



...

Traceback (most recent call last):
  File "/tmp/ipykernel_4456/2249140109.py", line 302, in train
    probs_train = model.predict_proba(X_train_imp)[:, 1]
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 391, in predict_proba
    return self._predict(X, output_type="probas")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 436, in _predict
    result = run_task(predict_task, "Predicting")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 960, in run_task
    result = future.result()
             ^^^^

Traceback (most recent call last):
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/wandb/agents/pyagent.py", line 296, in _run_job
    self._function()
  File "/tmp/ipykernel_4456/2249140109.py", line 302, in train
    probs_train = model.predict_proba(X_train_imp)[:, 1]
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 391, in predict_proba
    return self._predict(X, output_type="probas")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 436, in _predict
    result = run_task(predict_task, "Predicting")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/S

00:02 Fitting... Done!
00:00 Predicting... /

Traceback (most recent call last):
  File "/tmp/ipykernel_4456/2249140109.py", line 302, in train
    probs_train = model.predict_proba(X_train_imp)[:, 1]
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 391, in predict_proba
    return self._predict(X, output_type="probas")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 436, in _predict
    result = run_task(predict_task, "Predicting")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 960, in run_task
    result = future.result()
             ^^^^

Traceback (most recent call last):
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/wandb/agents/pyagent.py", line 296, in _run_job
    self._function()
  File "/tmp/ipykernel_4456/2249140109.py", line 302, in train
    probs_train = model.predict_proba(X_train_imp)[:, 1]
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 391, in predict_proba
    return self._predict(X, output_type="probas")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tabpfn_client/estimator.py", line 436, in _predict
    result = run_task(predict_task, "Predicting")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/giulio-casti/S

## Results of the experiment just run

One row per model, every cell the `mean ± std` across the evaluation seeds of the experiment currently selected in the *Dataset selection* cell (`tag`: `window`, `nowindow`, `noteam`, `nocompetitors`).

The std is the sample one (ddof=1) over the seeds, so a `± 0.000` cell means a single run rather than a model insensitive to the split — the `Seeds` column tells the two apart. Set `latex=True` to get the cells as `$mean \pm std$` for the paper.

In [12]:
# Results of the sweep that has just finished, for the experiment selected above.
# metrics_store is filled by make_train, one record per (model, tag, seed), so
# this cell needs the 35-run grid (7 models x 5 seeds) of that tag to be complete.
df_results = summarize_metrics(metrics_store, tag)

print(f"Experiment '{tag}' — mean ± std across seeds {config['sweep_settings']['parameters']['seed']['values']}")
print(df_results.to_string(index=False))

Experiment 'window' — mean ± std across seeds [1, 2, 3, 4, 5]
 Model  Seeds           AUC            F1     precision        recall      accuracy accuracy_train
    lr      5 0.790 ± 0.006 0.611 ± 0.005 0.579 ± 0.005 0.646 ± 0.012 0.736 ± 0.003  0.732 ± 0.001
    dt      5 0.781 ± 0.004 0.611 ± 0.004 0.541 ± 0.017 0.704 ± 0.030 0.713 ± 0.012  0.715 ± 0.008
    rf      5 0.798 ± 0.005 0.623 ± 0.005 0.591 ± 0.006 0.658 ± 0.009 0.744 ± 0.004  0.812 ± 0.002
   lgb      5 0.803 ± 0.005 0.634 ± 0.007 0.573 ± 0.007 0.709 ± 0.013 0.737 ± 0.005  0.758 ± 0.002
   svm      5 0.790 ± 0.005 0.560 ± 0.004 0.661 ± 0.004 0.485 ± 0.006 0.755 ± 0.002  0.759 ± 0.002
   mlp      5 0.794 ± 0.005 0.622 ± 0.005 0.562 ± 0.028 0.699 ± 0.043 0.727 ± 0.018  0.737 ± 0.017
tabpfn      1 0.808 ± 0.000 0.638 ± 0.000 0.584 ± 0.000 0.703 ± 0.000 0.744 ± 0.000  0.742 ± 0.000


## Metrics comparison across experiments

For each model, the table below compares the metrics of the `window` experiment against another experiment (`nowindow`, `nocompetitors`, `noteam`). 

Values are rounded to 2 decimals; `diff_%` is the percent change of the other experiment relative to `window` (`(other - window) / window * 100`).

Run all the experiments to populate `metrics_store` before executing the cells below.

### Comparison across window and nowindow experiments
You need to run both the "window" and "nowindow" runs before executing the following code, to ensure that shap_store and metrics_store contains the necessary data for both configurations

In [ ]:
df_cmp_nowindow = compare_metrics(metrics_store, "window", "nowindow")
print(df_cmp_nowindow.to_string(index=False))

In [ ]:
fig = plot_shap_comparison(shap_store)
plt.show()

In [ ]:
df_wilcoxon = compute_wilcoxon_table(shap_store)
print(df_wilcoxon.to_string(index=False))

### The 2x2: which leak moves the metrics

`window` is (aligned features, aligned label) and `nowindow` is (leaked, leaked), so the comparison above measures the two leaks summed together. The two tables below hold one axis fixed at a time:

- against `leaklabel`, only the target definition changes — and with it the base rate, from 0.321 to 0.405;
- against `leakfeat`, only the features change, at a constant base rate.

Read **AUC** across the label axis: it is insensitive to the base rate, while F1, precision and recall are not, since every model decides at p >= 0.5 without tuning a threshold. Run the `leaklabel` and `leakfeat` sweeps before executing these cells.

In [ ]:
df_cmp_leaklabel = compare_metrics(metrics_store, "window", "leaklabel",
                                   label_b="leaked label")
print(df_cmp_leaklabel.to_string(index=False))

In [ ]:
df_cmp_leakfeat = compare_metrics(metrics_store, "window", "leakfeat",
                                  label_b="leaked features")
print(df_cmp_leakfeat.to_string(index=False))

### Comparison across window and noteam experiments
You need to run both the "window" and "noteam" runs before executing the following code, to ensure that shap_store and metrics_store contains the necessary data for both configurations

In [ ]:
df_cmp_noteam = compare_metrics(metrics_store, "window", "noteam")
print(df_cmp_noteam.to_string(index=False))

### Comparison across window and nocompetitors experiments
You need to run both the "window" and "nocompetitors" runs before executing the following code, to ensure that shap_store and metrics_store contains the necessary data for both configurations

In [ ]:
df_cmp_nocompetitors = compare_metrics(metrics_store, "window", "nocompetitors")
print(df_cmp_nocompetitors.to_string(index=False))